In [ ]:
# flatten the mongo db data
from pymongo import MongoClient


def flatten_documents():
    # Connect to MongoDB
    client = MongoClient("mongodb://localhost:27017/")
    db = client["mastodon"]
    original_collection = db["trump"]
    new_collection = db["trump_flattened"]

    # Make sure the new collection is empty (Optional step)
    new_collection.drop()

    # Iterate over each document in the original collection
    for document in original_collection.find():
        # Extract descendants and ancestor
        descendants = document.pop("descendants", None)
        ancestors = document.pop("ancestors", None)

        # Create a flattened document
        flattened_document = document

        # Add the extracted fields back to the flattened document
        if descendants is not None:
            flattened_document["descendants"] = descendants
        if ancestors is not None:
            flattened_document["ancestors"] = ancestors

        # Insert the flattened document into the new collection
        new_collection.insert_one(flattened_document)


# Execute the flattening function
flatten_documents()


In [3]:
import pymongo
from pymongo import MongoClient


def flatten_and_insert_individual_records():
    # Connect to MongoDB
    client = MongoClient("mongodb://localhost:27017/")
    db = client["mastodon"]
    original_collection = db["biden"]
    new_collection = db["trump_flattened"]

    # Make sure the new collection is empty (Optional step)
    count = 0
    # Iterate over each document in the original collection
    for document in original_collection.find():
        # Extract and remove descendants and ancestors from the document
        descendants = document.pop("descendants", [])
        ancestors = document.pop("ancestors", [])

        # Insert the original document without descendants and ancestors into the new collection
        # if created_at is before "2024-09-21T21:52:00.494Z", set visited to True
        if document["created_at"] < "2024-09-21T21:52:00.494Z":
            document["visited"] = True
        else:
            document["visited"] = False
            count += 1
        # Combine all descendants and ancestors into a single list
        all_new_records = descendants + ancestors

        if not all_new_records:
            continue

        all_new_records = [{"_id": item["id"], **item} for item in all_new_records]
        # set visited to False
        for item in all_new_records:
            item["visited"] = False
        all_new_records.append(document)
        try:
            # Insert all new records into the new collection
            new_collection.insert_many(documents=all_new_records)
        except pymongo.errors.BulkWriteError as e:
            duplicate_count = sum(
                1 for error in e.details["writeErrors"] if error["code"] == 11000
            )

    print(f"Number of records with visited=False: {count}")


# Execute the function
flatten_and_insert_individual_records()


Number of records with visited=False: 0


In [4]:
# update ids from trump_stack to trump as visited
def update_visited():
    # Connect to MongoDB
    client = MongoClient("mongodb://localhost:27017/")
    db = client["mastodon"]
    trump_collection = db["trump_stack"]
    trump_flattened_collection = db["trump"]
    ids = trump_collection.distinct("_id")

    for id in ids:
        # Update the visited field to True for all documents in the trump collection
        trump_flattened_collection.update_one({"_id": id}, {"$set": {"visited": True}})


update_visited()
